# Project Leverage — AnyRBOnA53 data builder
Run all cells. The final cell downloads `anyrbona53.json`, which can be placed in `site/data/`.


In [ ]:
!pip -q install nflreadpy polars pyarrow


In [ ]:
from pathlib import Path
script = r'''from __future__ import annotations

import argparse
import json
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import nflreadpy as nfl
import polars as pl


def first_existing(df: pl.DataFrame, names: list[str]) -> str | None:
    return next((name for name in names if name in df.columns), None)


def normalize_name(value: str | None) -> str:
    return "".join(ch.lower() for ch in (value or "") if ch.isalnum())


def fantasy_points(row: dict[str, Any], mode: str = "half") -> float:
    ppr = float(row.get("fantasy_points_ppr") or 0)
    std = float(row.get("fantasy_points") or 0)
    rec = float(row.get("receptions") or 0)
    if mode == "ppr":
        return ppr if ppr else std + rec
    if mode == "standard":
        return std if std else max(0.0, ppr - rec)
    return std + 0.5 * rec if std else ppr - 0.5 * rec


def rate(rows: list[dict[str, Any]], predicate) -> dict[str, float | int]:
    selected = [row for row in rows if predicate(row)]
    return {
        "n": len(selected),
        "hit": sum(bool(row["hit"]) for row in selected) / (len(selected) or 1),
    }


def build(seasons: list[int], output: Path) -> None:
    needed = sorted(set([min(seasons) - 1, *seasons]))
    stats = nfl.load_player_stats(needed, summary_level="week")
    if not isinstance(stats, pl.DataFrame):
        stats = pl.DataFrame(stats)

    position_col = first_existing(stats, ["position", "position_group"])
    team_col = first_existing(stats, ["recent_team", "team", "team_abbr"])
    id_col = first_existing(stats, ["player_id", "gsis_id"])
    name_col = first_existing(stats, ["player_display_name", "player_name"])
    carry_col = first_existing(stats, ["carries", "rushing_attempts"])
    target_col = first_existing(stats, ["targets"])
    if not all([position_col, team_col, id_col, name_col, carry_col, target_col]):
        raise RuntimeError(f"Unexpected player stats schema: {stats.columns}")

    stats = stats.filter(
        pl.col("season").is_in(needed)
        & pl.col("week").is_between(1, 18)
        & pl.col(position_col).is_in(["RB", "WR"])
    )
    stat_rows = stats.to_dicts()

    injury_frames: list[pl.DataFrame] = []
    injury_seasons: list[int] = []
    for season in seasons:
        try:
            frame = nfl.load_injuries(season)
            if not isinstance(frame, pl.DataFrame):
                frame = pl.DataFrame(frame)
            injury_frames.append(frame)
            injury_seasons.append(season)
        except Exception as exc:
            print(f"Warning: injury data unavailable for {season}: {exc}")
    injuries = pl.concat(injury_frames, how="diagonal_relaxed") if injury_frames else pl.DataFrame()

    players: dict[tuple[int, str, str], dict[str, Any]] = {}
    for row in stat_rows:
        season = int(row["season"])
        pos = str(row[position_col])
        pid = str(row[id_col] or row[name_col])
        key = (season, pos, pid)
        player = players.setdefault(
            key,
            {
                "season": season,
                "pos": pos,
                "id": pid,
                "name": row[name_col] or pid,
                "team": row[team_col] or "",
                "total": 0.0,
                "weeks": [],
            },
        )
        fp = fantasy_points(row)
        carries = float(row.get(carry_col) or 0)
        targets = float(row.get(target_col) or 0)
        team = row.get(team_col) or player["team"]
        player["team"] = team
        player["total"] += fp
        player["weeks"].append(
            {
                "week": int(row["week"]),
                "fp": fp,
                "touches": carries + targets,
                "carries": carries,
                "targets": targets,
                "team": team,
            }
        )

    prior_rank: dict[tuple[int, str, str], int] = {}
    for season in seasons:
        for pos in ("RB", "WR"):
            cohort = sorted(
                [p for p in players.values() if p["season"] == season - 1 and p["pos"] == pos],
                key=lambda p: p["total"],
                reverse=True,
            )
            for rank, player in enumerate(cohort, 1):
                prior_rank[(season, pos, player["id"])] = rank

    weekly_rank: dict[tuple[int, int, str, str], int] = {}
    for season in seasons:
        for week in range(1, 19):
            for pos in ("RB", "WR"):
                entries = []
                for p in players.values():
                    if p["season"] != season or p["pos"] != pos:
                        continue
                    current = next((w for w in p["weeks"] if w["week"] == week), None)
                    if current:
                        entries.append((p["id"], current["fp"]))
                for rank, (pid, _) in enumerate(sorted(entries, key=lambda x: x[1], reverse=True), 1):
                    weekly_rank[(season, week, pos, pid)] = rank

    injuries_by_team_week: dict[tuple[int, int, str], list[dict[str, Any]]] = defaultdict(list)
    if injuries.height:
        ipos = first_existing(injuries, ["position"])
        iteam = first_existing(injuries, ["team"])
        iid = first_existing(injuries, ["gsis_id", "player_id"])
        iname = first_existing(injuries, ["full_name", "player_name"])
        if ipos and iteam:
            for row in injuries.to_dicts():
                if str(row.get(ipos) or "").upper() != "RB":
                    continue
                season, week = int(row.get("season") or 0), int(row.get("week") or 0)
                team = str(row.get(iteam) or "")
                if season not in seasons or not team or not 1 <= week <= 18:
                    continue
                status = str(row.get("report_status") or "").lower()
                practice = str(row.get("practice_status") or "").lower()
                unavailable = any(x in status for x in ["out", "doubtful", "reserve", "inactive"])
                uncertain = "questionable" in status or any(x in practice for x in ["did not participate", "limited"])
                if not unavailable and not uncertain:
                    continue
                name = str(row.get(iname) or "") if iname else ""
                rid = str(row.get(iid) or normalize_name(name)) if iid else normalize_name(name)
                injuries_by_team_week[(season, week, team)].append(
                    {
                        "id": rid,
                        "name": name or rid,
                        "status": status,
                        "practice": practice,
                        "unavailable": unavailable,
                    }
                )

    def teammate_prior_usage(season: int, week: int, team: str, injury: dict[str, Any]) -> float:
        best = 0.0
        for p in players.values():
            if p["season"] != season or p["pos"] != "RB" or p["team"] != team:
                continue
            if p["id"] != injury["id"] and normalize_name(p["name"]) != normalize_name(injury["name"]):
                continue
            prev = sorted([w for w in p["weeks"] if w["week"] < week], key=lambda w: w["week"], reverse=True)[:3]
            if prev:
                best = max(best, sum(w["touches"] for w in prev) / len(prev))
        return best

    ranges = {"RB": (35, 60), "WR": (60, 90)}
    usable = {"RB": 24, "WR": 36}
    obs: dict[str, list[dict[str, Any]]] = {"RB": [], "WR": []}

    for p in players.values():
        if p["season"] not in seasons:
            continue
        pr = prior_rank.get((p["season"], p["pos"], p["id"]))
        lo, hi = ranges[p["pos"]]
        if not pr or not lo <= pr <= hi:
            continue
        weeks = sorted(p["weeks"], key=lambda w: w["week"])
        for current in weeks:
            prior = [w for w in weeks if w["week"] < current["week"]]
            last3 = prior[-3:]
            avg3 = sum(w["touches"] for w in last3) / len(last3) if last3 else 0.0
            target_avg3 = sum(w["targets"] for w in last3) / len(last3) if last3 else 0.0
            last = prior[-1]["touches"] if prior else 0.0
            rank = weekly_rank.get((p["season"], current["week"], p["pos"], p["id"]), 999)
            vacated = 0.0
            injury_names: list[str] = []
            if p["pos"] == "RB":
                for injury in injuries_by_team_week.get((p["season"], current["week"], current["team"]), []):
                    if injury["id"] == p["id"] or normalize_name(injury["name"]) == normalize_name(p["name"]):
                        continue
                    usage = teammate_prior_usage(p["season"], current["week"], current["team"], injury)
                    weighted = usage * (1.0 if injury["unavailable"] else 0.35)
                    vacated += weighted
                    if weighted >= 2:
                        injury_names.append(f"{injury['name']} ({injury['status'] or injury['practice']})")
            workload_signal = last >= 10 or avg3 >= 8
            injury_signal = vacated >= 6
            receiving_signal = target_avg3 >= 3
            delta = min(
                100,
                round(
                    min(30, avg3 * 2)
                    + min(12, max(0, last - avg3) * 3)
                    + min(35, vacated * 3)
                    + min(12, target_avg3 * 3)
                    + (10 if injury_signal and workload_signal else 0)
                ),
            )
            obs[p["pos"]].append(
                {
                    "season": p["season"],
                    "week": current["week"],
                    "name": p["name"],
                    "team": current["team"],
                    "priorRank": pr,
                    "fp": current["fp"],
                    "weekRank": rank,
                    "hit": rank <= usable[p["pos"]],
                    "last": last,
                    "avg3": avg3,
                    "targetAvg3": target_avg3,
                    "vacated": vacated,
                    "injuryNames": injury_names,
                    "workloadSignal": workload_signal,
                    "injurySignal": injury_signal,
                    "receivingSignal": receiving_signal,
                    "combinedSignal": workload_signal or injury_signal,
                    "opportunityDelta": delta,
                }
            )

    def summarize(rows: list[dict[str, Any]], pos: str) -> dict[str, Any]:
        players_seen = {(r["season"], r["name"]) for r in rows}
        player_hits = {(r["season"], r["name"]) for r in rows if r["hit"]}
        counts: dict[tuple[int, str], int] = defaultdict(int)
        for row in rows:
            counts[(row["season"], row["name"])] += int(row["hit"])
        return {
            "pos": pos,
            "observations": len(rows),
            "players": len(players_seen),
            "hits": sum(int(r["hit"]) for r in rows),
            "hitRate": sum(int(r["hit"]) for r in rows) / (len(rows) or 1),
            "playerHitRate": len(player_hits) / (len(players_seen) or 1),
            "avgUsableWeeks": sum(counts.values()) / (len(counts) or 1),
            "workload": rate(rows, lambda r: r["workloadSignal"]),
            "injury": rate(rows, lambda r: r["injurySignal"]),
            "combined": rate(rows, lambda r: r["combinedSignal"]),
            "neither": rate(rows, lambda r: not r["combinedSignal"]),
            "highDelta": rate(rows, lambda r: r["opportunityDelta"] >= 50),
            "lowDelta": rate(rows, lambda r: r["opportunityDelta"] < 50),
        }

    examples = sorted(
        [r for r in obs["RB"] if r["injurySignal"]],
        key=lambda r: (r["opportunityDelta"], r["fp"]),
        reverse=True,
    )[:12]

    payload = {
        "metadata": {
            "built_at": datetime.now(timezone.utc).isoformat(),
            "source": "nflverse via nflreadpy",
            "injury_seasons": injury_seasons,
            "methodology_version": "0.1.0",
        },
        "seasons": seasons,
        "result": {
            "RB": summarize(obs["RB"], "RB"),
            "WR": summarize(obs["WR"], "WR"),
            "examples": examples,
        },
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote {output} ({output.stat().st_size / 1024:.1f} KiB)")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--seasons", default="2019-2024")
    parser.add_argument("--output", default="site/data/anyrbona53.json")
    args = parser.parse_args()
    start, end = map(int, args.seasons.split("-"))
    build(list(range(start, end + 1)), Path(args.output))
'''
Path("build_anyrb_data.py").write_text(script)
!python build_anyrb_data.py --seasons 2019-2024 --output anyrbona53.json


In [ ]:
from google.colab import files
files.download("anyrbona53.json")
